4. Add _metadata.file_name and _metadata.file_path to your ingestion query and use them to prove which source file each row came from. 

In [0]:

CREATE OR REPLACE TABLE cyntexa_dev.default.customer_jsonread_metadata
USING DELTA AS
SELECT 
customer_id,
personal.first_name as first_name,
personal.last_name as last_name,
personal.gender as gender,
contact.email as email,
contact.phone as phone,
contact.address.city as city,
contact.address.state as state,
contact.address.country as country,
employment.company AS company,
employment.department AS department,
employment.designation AS designation,

_metadata.file_name AS source_file_name,
_metadata.file_path AS source_file_path

FROM
read_files('/Volumes/cyntexa_dev/sales/raw/*.json', format=>'json');

In [0]:
SELECT
    customer_id,
    _metadata
FROM read_files(
    '/Volumes/cyntexa_dev/sales/raw/customers_nested_jsonlines.json',
    format => 'json'
);

5. Create an Iceberg table from the same source data and compare its DESCRIBE DETAIL output (format, location) to the Delta version. 


In [0]:
CREATE OR REPLACE TABLE cyntexa_dev.default.customer_nestedjson_iceberg
USING iceberg
AS
SELECT
    customer_id,

    personal.first_name AS first_name,
    personal.last_name AS last_name,
    personal.gender AS gender,

    contact.email AS email,
    contact.phone AS phone,
    contact.address.city AS city,
    contact.address.state AS state,
    contact.address.country AS country,

    employment.company AS company,
    employment.department AS department,
    employment.designation AS designation,

    _metadata.file_name AS source_file_name,
    _metadata.file_path AS source_file_path

FROM read_files(
    '/Volumes/cyntexa_dev/sales/raw/*.json',
    format => 'json'
);

In [0]:
DESCRIBE DETAIL cyntexa_dev.default.customer_nestedjson_iceberg;

6. (Data Analyst) Write a query using the metadata columns to build a 'records per source file' audit report — useful for verifying a vendor's daily file drop. 

In [0]:
SELECT
source_file_name AS file_name,
source_file_path AS file_path,
COUNT(source_file_path) AS num_records
FROM cyntexa_dev.default.customer_nestedjson_iceberg
GROUP BY source_file_name,
source_file_path
ORDER BY num_records DESC;
